# Environment Setup

In [1]:
!git clone https://github.com/PacktPublishing/Distributed-AI-Systems

Cloning into 'Distributed-AI-Systems'...
remote: Enumerating objects: 340, done.
remote: Counting objects: 100% (340/340), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 340 (delta 69), reused 332 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (340/340), 1.63 MiB | 10.18 MiB/s, done.
Resolving deltas: 100% (69/69), done.


In [2]:
!pip install -r /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/requirements.txt

# Step 1: inspect GPU hardware

In [3]:
!python /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/check_cuda.py


CUDA and GPU Information
CUDA available: True
CUDA version: 12.8
cuDNN version: 91002
Number of GPUs: 2

GPU 0: Tesla T4
  Total memory: 14.6 GB
  Compute capability: 7.5
  Multiprocessors: 40

GPU 1: Tesla T4
  Total memory: 14.6 GB
  Compute capability: 7.5
  Multiprocessors: 40

Note: For detailed PCIe and NVLink topology, run:
  nvidia-smi topo -m
  nvidia-smi --query-gpu=name,memory.total,pcie.link.gen.max,pcie.link.width.max --format=csv


# Step 2: inspect hardware topology

In [4]:
!nvidia-smi topo -m

	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


In [5]:
!nvidia-smi --query-gpu=name,memory.total,pcie.link.gen.max,pcie.link.width.max --format=csv


name, memory.total [MiB], pcie.link.gen.max, pcie.link.width.max
Tesla T4, 15360 MiB, 3, 16
Tesla T4, 15360 MiB, 3, 16


# Step 3: measure Single-GPU memory bandwidth


In [6]:
!python /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/bandwidth_test.py

Bandwidth: 108.19 GB/s


# Step 4: benchmark Inter-GPU communication

In [7]:
%%writefile /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/allreduce_microbench.py
import os
import torch
import torch.distributed as dist
import time

def allreduce_microbench():
    # Initialize the distributed environment if not already initialized
    if not dist.is_initialized():
        rank = int(os.environ.get("RANK", "0"))
        world_size = int(os.environ.get("WORLD_SIZE", "1"))
        master_addr = os.environ.get("MASTER_ADDR", "localhost")
        master_port = os.environ.get("MASTER_PORT", "29500")

        backend = 'nccl' if torch.cuda.is_available() else 'gloo'

        print(f"Rank {rank}: Initializing process group with backend={backend}")
        dist.init_process_group(backend=backend,
                                rank=rank,
                                world_size=world_size,
                                init_method=f"tcp://{master_addr}:{master_port}")

    if torch.cuda.is_available():
        torch.cuda.set_device(dist.get_rank() % torch.cuda.device_count())

    device = torch.device(f'cuda:{dist.get_rank() % torch.cuda.device_count()}' if torch.cuda.is_available() else 'cpu')
    tensor_size = 1024 * 1024 * 4 # Example: 4MB float tensor
    t = torch.ones(tensor_size, dtype=torch.float32, device=device) * (dist.get_rank() + 1)

    print(f"Rank {dist.get_rank()}: Initial tensor sum: {t.sum().item()}")

    start_time = time.time()

    dist.all_reduce(t)

    end_time = time.time()
    elapsed_time_ms = (end_time - start_time) * 1000

    print(f"Rank {dist.get_rank()}: All-reduce completed in {elapsed_time_ms:.2f} ms")
    print(f"Rank {dist.get_rank()}: All-reduced tensor sum: {t.sum().item()}")

if __name__ == "__main__":
    allreduce_microbench()



Overwriting /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/allreduce_microbench.py


In [8]:
!torchrun --nproc_per_node=2 /kaggle/working/Distributed-AI-Systems/chapter2-gpu-hardware-networking-and-parallelism-strategies/code/allreduce_microbench.py

W0815 05:54:35.453000 76 torch/distributed/run.py:852] 
W0815 05:54:35.453000 76 torch/distributed/run.py:852] *****************************************
W0815 05:54:35.453000 76 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0815 05:54:35.453000 76 torch/distributed/run.py:852] *****************************************
Rank 1: Initializing process group with backend=nccl
Rank 0: Initializing process group with backend=nccl
Rank 0: Initial tensor sum: 4194304.0
Rank 1: Initial tensor sum: 8388608.0
Rank 1: All-reduce completed in 432.26 msRank 0: All-reduce completed in 432.44 ms

Rank 1: All-reduced tensor sum: 12582912.0Rank 0: All-reduced tensor sum: 12582912.0

[rank0]:[W815 05:54:38.472562201 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program e